# Elise Failure Audit

Find and inspect the worst greedy evaluation episodes for a saved SolarSystemLander checkpoint.

In [ ]:
# cell: colab-setup

!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.notebook.colab import setup_colab

COLAB = setup_colab()

In [ ]:
# cell: audit-setup; requires: colab-setup

import json

import torch

from hpo.objective import ObjectiveConfig
from hpo.solar_system_lander.environment import EnvFactory, World

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OBSERVATION_MODE = "10d"
DATABASE = f"solar_system_lander_{OBSERVATION_MODE}_elise_stp"

WORLD_MIX = {World.MERCURY: 1, World.VENUS: 4, World.EARTH: 4, World.MOON: 1, World.MARS: 1}
ENV_FACTORY = EnvFactory(OBSERVATION_MODE, world_mix=WORLD_MIX)

CHECKPOINT_DIR = COLAB.drive_study_dir / "best_checkpoints" / DATABASE
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_eval_checkpoint.pt"
CHECKPOINT_METADATA_PATH = CHECKPOINT_DIR / "best_eval_checkpoint.json"
VIDEO_DIR = COLAB.drive_study_dir / "videos" / DATABASE / "failure_audit"

OBJECTIVE_CFG = ObjectiveConfig(
    environment_factory=ENV_FACTORY,
    num_envs=22,
    eval_episodes=10,
    device=device,
)

metadata = json.loads(CHECKPOINT_METADATA_PATH.read_text())
print(f"checkpoint: {CHECKPOINT_PATH}")
print(f"source score: {metadata['score']:.1f}")
print(f"trial: {metadata['trial_number']}")
metadata["world_scores"]

In [ ]:
# cell: worst-checkpoint-episodes; requires: audit-setup

from hpo.evaluation.checkpoint_robustness import checkpoint_scores, score_summary

_EPISODES = 200
_WORST_N = 20

scores = checkpoint_scores(CHECKPOINT_PATH, OBJECTIVE_CFG, episodes=_EPISODES)

if OBJECTIVE_CFG.eval_seed is None:
    scores["seed"] = None
else:
    scores["seed"] = scores["episode"].astype(int) + OBJECTIVE_CFG.eval_seed

worst_checkpoint_episodes = scores.sort_values("score").head(_WORST_N).reset_index(drop=True)

display(score_summary(scores).round(1))
display(worst_checkpoint_episodes)

In [ ]:
# cell: worst-video-jobs; requires: worst-checkpoint-episodes

import pandas as pd

from hpo.evaluation.video import video_conditions_table

_VIDEO_N = 5

worst_video_jobs = worst_checkpoint_episodes.head(_VIDEO_N).copy()

_condition_rows = []
for nr, row in enumerate(worst_video_jobs.itertuples(index=False)):
    _table = video_conditions_table(ENV_FACTORY, [row.world], [int(row.seed)]).drop(columns=["nr"])
    _table.insert(0, "nr", nr)
    _table.insert(3, "score", row.score)
    _condition_rows.append(_table)

worst_video_conditions = pd.concat(_condition_rows, ignore_index=True)
display(worst_video_conditions)

In [ ]:
# cell: record-worst-videos; requires: worst-video-jobs

from hpo.evaluation.lander_rendering import LanderOverlay, world_colors
from hpo.evaluation.video import record_checkpoint_video

_WORLDS = [World.MERCURY, World.VENUS, World.EARTH, World.MOON, World.MARS]
_RENDER_COLORS = {str(world): color for world, color in zip(_WORLDS, world_colors(_WORLDS))}

worst_video_paths = [
    record_checkpoint_video(
        checkpoint_path=CHECKPOINT_PATH,
        environment_factory=ENV_FACTORY,
        world=row.world,
        seed=int(row.seed),
        output_dir=VIDEO_DIR,
        device=device,
        max_steps=OBJECTIVE_CFG.eval_max_steps,
        render_colors=_RENDER_COLORS[str(row.world)],
        render_overlay=LanderOverlay(),
    )
    for row in worst_video_jobs.itertuples(index=False)
]

worst_video_paths

In [ ]:
# cell: display-worst-video; requires: record-worst-videos

from hpo.evaluation.video import display_video

display_video(worst_video_paths, 0)